<a href="https://colab.research.google.com/github/ArtyomShabunin/SMOPA-25/blob/main/lesson_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://prana-system.com/files/110/rds_color_full.png" alt="tot image" width="300"  align="center"/> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://mpei.ru/AboutUniverse/OficialInfo/Attributes/PublishingImages/logo1.jpg" alt="mpei image" width="200" align="center"/>
<img src="https://mpei.ru/Structure/Universe/tanpe/structure/tfhe/PublishingImages/tot.png" alt="tot image" width="100"  align="center"/>

---

# **Системы машинного обучения и предиктивной аналитики в тепловой и возобновляемой энергетике**  

# ***Практические занятия***


---



# Занятие №4
# Поиск аномалий методами глубокого обучения
**11 марта 2026г.**

## Импорты

In [ ]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import seaborn as sns
sns.set_theme(style="whitegrid", rc={'figure.figsize':(15,6)})

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
import json

import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objs as go

pd.options.mode.chained_assignment = None

from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))

## Минимум PyTorch

### Что такое PyTorch
**PyTorch** — это библиотека для машинного обучения и глубоких нейросетей на Python.
С ее помощью удобно описывать модели, обучать их на CPU/GPU и быстро проводить эксперименты.

### Почему PyTorch часто выбирают
- понятный и 'питоновский' стиль кода;
- удобно отлаживать модели шаг за шагом (динамический граф вычислений);
- сильная экосистема для deep learning (torchvision, torchaudio и др.);
- хорошо подходит для исследовательских и учебных задач, где часто меняется архитектура.

### Что такое `torch.Tensor`
`torch.Tensor` — это базовый тип данных в PyTorch: многомерный числовой массив (аналог `numpy.ndarray`), с которым работает нейросеть.

У тензора есть ключевые свойства:
- `shape` — форма (размерности), например `(batch_size, n_features)`;
- `dtype` — тип данных (`float32`, `int64` и т.д.);
- `device` — где лежит тензор (`cpu` или `cuda`);
- `requires_grad` — нужно ли считать градиенты для обучения.

В уроке автокодировщик принимает на вход тензор признаков и возвращает тензор реконструированных признаков той же формы.

Ниже — короткий практический блок с операциями, которые понадобятся дальше.


Создание тензора из списка

In [ ]:
x = torch.tensor([1, 2, 3])
x

In [ ]:
x.dtype

Задание типа данных (`dtype`)

In [ ]:
x = torch.tensor([1, 2, 3], dtype=torch.float32)
x.dtype

In [ ]:
x.to(torch.int32)

Создание тензора с нулями и единицами

In [ ]:
zeros = torch.zeros(3, 3)  # Матрица 3x3 из нулей
ones = torch.ones(2, 2)    # Матрица 2x2 из единиц
rand = torch.rand(2, 3)    # Матрица 2x3 со случайными числами

zeros, ones, rand

Основные свойства тензоров

In [ ]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])

print(x.shape)  # Размерность (2, 3)
print(x.dtype)  # Тип данных (int64)
print(x.device) # Где хранится (CPU/GPU)

Перевод на GPU

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.tensor([1, 2, 3], device=device)
print(x.device)  # cuda (если есть GPU)

Преобразование между NumPy и PyTorch

In [ ]:
# Из NumPy в PyTorch
np_array = np.array([1, 2, 3])
torch_tensor = torch.from_numpy(np_array)

# Из PyTorch в NumPy
numpy_array = torch_tensor.numpy()

torch_tensor, numpy_array

In [ ]:
torch_tensor = torch.from_numpy(np_array)
torch_tensor.detach().numpy() # detach() используется в PyTorch, чтобы отключить градиенты у тензора.
                              # Это позволяет создать новый тензор, который не участвует в вычислении градиентов,
                              # но содержит те же значения, что и исходный тензор

Операции с тензорами

In [ ]:
x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5, 6])

print(x + y)  # Сложение
print(x * y)  # Поэлементное умножение
print(x @ y)  # Скалярное произведение

Автоматическое вычисление градиентов (`requires_grad=True`)

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x**2  # y = x^2

y.backward()  # Вычисляем градиент
print(x.grad)  # dy/dx = 2*x = 4

Добавление размерности

In [ ]:
# метод unsqueeze
x = torch.tensor([1, 2, 3])  # Размерность: (3,)
x_unsqueezed = x.unsqueeze(0)  # Добавляем ось по нулевому измерению
x_unsqueezed, x_unsqueezed.shape  # torch.Size([1, 3])

In [ ]:
# метод view
x = torch.tensor([1, 2, 3])  # Размерность: (3,)
x_reshaped = x.view(1, 3)  # То же самое, что unsqueeze(0)
x_reshaped, x_reshaped.shape  # torch.Size([1, 3])

In [ ]:
# индексация
x = torch.tensor([1, 2, 3])  # Размерность: (3,)
x_expanded = x[None, :]
print(x_expanded.shape)  # torch.Size([1, 3])

Удаление размерности

In [ ]:
# метод squeeze
x = torch.tensor([[1, 2, 3]])  # Размерность (1, 3)
x_squeezed = x.squeeze(0)  # Удаляем нулевую ось
print(x_squeezed.shape)  # torch.Size([3])

In [ ]:
# индексация
x = torch.tensor([[1, 2, 3]])  # Размерность (1, 3)
x_squeezed = x_expanded[0]
print(x_squeezed.shape)  # torch.Size([3])

#### Что такое `broadcasting`

**Broadcasting** — это механизм, который позволяет выполнять арифметические операции над тензорами (или массивами) разной формы без явного копирования данных: оси с размером `1` логически растягиваются до нужного размера.

Этот механизм есть и в **NumPy**, и в **PyTorch** (правила одинаковые по смыслу).

Если сравнивать размерности справа налево, broadcasting срабатывает, когда для каждой пары размерностей выполняется одно из условий:
- размерности равны;
- одна из размерностей равна `1`;
- у одного тензора размерность отсутствует (мысленно считаем её равной `1`).


#### 1) Где broadcasting работает автоматически


In [ ]:
# (2, 3) + (3,) -> (2, 3)
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])      # shape: (2, 3)
b = torch.tensor([10., 20., 30.])      # shape: (3,)

out = A + b
print('out shape:', out.shape)
print(out)


Вектор длины 3 автоматически применяется к каждому ряду матрицы `(2, 3)`.


In [ ]:
# (3, 1) * (1, 4) -> (3, 4)
x = torch.tensor([[1.], [2.], [3.]])    # shape: (3, 1)
y = torch.tensor([[10., 20., 30., 40.]]) # shape: (1, 4)

out = x * y
print('out shape:', out.shape)
print(out)


Оси с `1` расширяются автоматически, поэтому получаем таблицу всех попарных произведений.


#### 2) Где broadcasting не работает и нужно явное приведение


In [ ]:
# Хотим умножить каждую строку матрицы на свой коэффициент
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])      # shape: (2, 3)
k = torch.tensor([10., 100.])         # shape: (2,)

# out = A * k
# Ошибка без явного приведения:
# RuntimeError: The size of tensor a (3) must match the size of tensor b (2) ...

out = A * k.unsqueeze(1)                # (2,) -> (2, 1)
print('out shape:', out.shape)
print(out)


`unsqueeze(1)` делает из `k` столбец, после чего коэффициенты корректно применяются по строкам.


In [ ]:
# Вычесть среднее каждой строки из элементов этой строки
A = torch.tensor([[1., 2., 3.],
                  [10., 20., 30.]])   # shape: (2, 3)
row_mean = A.mean(dim=1)              # shape: (2,)

# centered = A - row_mean
# Ошибка без явного приведения:
# RuntimeError: The size of tensor a (3) must match the size of tensor b (2) ...

centered = A - row_mean.unsqueeze(1)   # (2,) -> (2, 1)
print('centered shape:', centered.shape)
print(centered)


Та же идея: для операций по строкам часто нужно вручную добавить ось столбца.


## Подготовка данных для поиска аномалий


In [ ]:
# import gdown
# import warnings
# warnings.filterwarnings('ignore')
# gdown.download('https://drive.google.com/uc?id=1uEm4d4rbh1RAPw4JOHJiHzw5y_m3OSj3', verify=False)

data = pd.read_parquet("./data.gzip")

# signals = [
#     'GTA1.DBinPU.Aldi', 'GTA1.DBinPU.Alvna', 'GTA1.DBinPU.Alzzo',
#     'GTA1.DBinPU.Bo', 'GTA1.DBinPU.fi', 'GTA1.DBinPU.nst',
#     'GTA1.DBinPU.ntk', 'GTA1.DBinPU.P', 'GTA1.DBinPU.Pk', 'GTA1.DBinPU.Pvh',
#     'GTA1.DBinPU.Qtg', 'GTA1.DBinPU.Tk', 'GTA1.DBinPU.Tn', 'GTA1.DBinPU.Tt']

# data = data.loc[:, signals]

In [ ]:
data.head()

In [ ]:
data.shape

Чтение файла с описанием сигналов

In [ ]:
with open(f'./option_0/description.json', 'r', encoding = "utf-8") as f:
    description = json.load(f)

Составим словарь для трактовки наименований сигналов

In [ ]:
kks_to_description = {param['real_kks']: f"{param['description']}, [{param['unit']}]"
for param in description if param['real_kks'] in data.columns}

description_to_kks = { f"{param['description']}, [{param['unit']}]": param['real_kks']
for param in description if param['real_kks'] in data.columns}

### Деление на обучающую и тестовую выборки

In [ ]:
shuffled_data = data.sample(frac=1)

data_train = shuffled_data.iloc[:round(shuffled_data.shape[0]*0.8), :]
data_test = shuffled_data.iloc[round(shuffled_data.shape[0]*0.8):, :]

Добавим шум на половину тестовых данных и отметим их как аномальные

In [ ]:
def generate_binary_matrix(rows, cols, prob_ones=0.4):
    matrix = np.zeros((rows, cols), dtype=int)  # Создаем матрицу из нулей

    for i in range(rows):
        # Генерируем случайные 0 и 1 с заданной вероятностью (без гарантированной 1)
        row = np.random.choice([0, 1], size=cols, p=[1 - prob_ones, prob_ones])

        # Если в строке нет 1, вставляем её в случайное место
        if not np.any(row):
            row[np.random.randint(0, cols)] = 1

        matrix[i] = row  # Записываем строку в матрицу

    return matrix

In [ ]:
NUMBER_OF_ANOMALY_POINTS = round(data_test.shape[0]*0.2)

data_test["anomaly"] = 0
data_test.loc[
    data_test.iloc[:NUMBER_OF_ANOMALY_POINTS].index, ["anomaly"]] = 1

mask = generate_binary_matrix(NUMBER_OF_ANOMALY_POINTS, data.shape[1], prob_ones=0.3)

bias = mask * np.random.choice(
    [-0.05, -0.04, -0.03, -0.02, -0.01, 0.01, 0.02, 0.03, 0.04, 0.05],
    size=[NUMBER_OF_ANOMALY_POINTS, data.shape[1]],
    p=np.full(10,0.1))

data_test.loc[
    data_test.iloc[:NUMBER_OF_ANOMALY_POINTS].index,
    data_test.columns != "anomaly"] = data_test[data_test["anomaly"] == 1].loc[:,data_test.columns != "anomaly"] * (1 + bias)

In [ ]:
params_dropdown = widgets.Dropdown(
    options=data_test.columns,
    description='Параметр:',
    disabled=False,
    value=None
)

out = widgets.Output()
display(out)

with out:
    display(params_dropdown)

@out.capture()
def params_dropdown_eventhandler(change):

    clear_output()
    display(params_dropdown)

    fig, axes = plt.subplots(1, 1, figsize=(15,5))
    plt.title(f"Тестовая выборка\n{change.new} - {kks_to_description [change.new]}")

    data_test[data_test["anomaly"] == 0][change.new].plot(style='.', label="Нормальные данные");
    data_test[data_test["anomaly"] == 1][change.new].plot(style='.', label="Зашумленные данные");

    # plt.plot(scat_1[change.new], 'r.', markersize=5, label='Аномалии')
    # plt.plot(scat_0[change.new], 'g.', markersize=5, label='Норма')
    plt.legend()
    display(fig)

params_dropdown.observe(params_dropdown_eventhandler, names='value')

### Нормализация или стандартизация данных

In [ ]:
scaler = preprocessing.MinMaxScaler() # нормализация данных
# scaler = preprocessing.StandardScaler() # стандартизация данных

X_train = pd.DataFrame(
    scaler.fit_transform(data_train),
    columns=data_train.columns,
    index=data_train.index)

X_test = pd.DataFrame(
    scaler.transform(data_test[data.columns]),
    columns=data.columns,
    index=data_test.index)

X_train.describe()

## Полносвязные сети и автокодировщик

### Полносвязная нейронная сеть (Fully Connected Layer, FC Layer)

Персептрон — это **базовая модель** искусственной нейронной сети, предложенная Фрэнком Розенблаттом в 1958 году. Это **самый простой вид нейросети**, который является основой для современных **полносвязных нейронных сетей**.

---
### **Персептрон**
**Структура персептрона**

<img src="https://neerc.ifmo.ru/wiki/images/a/a5/%D0%98%D1%81%D0%BA%D1%83%D1%81%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BD%D0%B5%D0%B9%D1%80%D0%BE%D0%BD_%D1%81%D1%85%D0%B5%D0%BC%D0%B0.png" alt="perceptron image" width="500"/>

Классический **одиночный персептрон** состоит из:  
- **Входного слоя** $ x_1, x_2, ..., x_n $ — входные признаки  
- **Весов** $ w_1, w_2, ..., w_n $ — коэффициенты, определяющие важность входных признаков  
- **Смещения** (bias) $ b $ — позволяет модели лучше подстраиваться  
- **Сумматора** — вычисляет взвешенную сумму входов:  
  $$ z = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b $$
- **Активационной функции** $ f(z) $, например, пороговой (step function):  
  $$
  y =
  \begin{cases}
  1, & z \geq 0 \
  0, & z < 0
  \end{cases}
  $$
  
Персептрон работает как **линейный классификатор**, разделяя данные на два класса.

---
### **Многослойный персептрон**

- **Одиночный персептрон** — это всего **один полносвязный слой**.  
- **Многослойный персептрон (MLP, Multi-Layer Perceptron)** — это **полносвязная нейросеть**, состоящая из **нескольких персептронов** в скрытых слоях.  
- В отличие от обычного персептрона, **MLP может моделировать нелинейные зависимости** благодаря использованию **нелинейных активационных функций** (ReLU, Sigmoid, Tanh).

<img src="https://lh5.googleusercontent.com/proxy/_NrlkWwft7zy_-C-fxPidblqeLH0wZXd5Vj2MzsM0twc0mp72IPtSrpijoQ-4rfx9BSBbHbFAkXo7BaMXKdAaUhg2tGH" alt="mlp image" width="400"/>

---

**Формула двухслойного персептрона через матричное умножение**  

Рассмотрим **двухслойный персептрон** (один скрытый слой + выходной слой).  

**Обозначения:**  
- $ X $ — входной вектор (размерность $ (d, n) $, где $ d $ — количество примеров, $ n $ — число входных признаков).  
- $ W_1 $ — матрица весов первого слоя (размерность $ (n, h) $, где $ h $ — число нейронов в скрытом слое).  
- $ b_1 $ — вектор смещений первого слоя (размерность $ (1, h) $).  
- $ W_2 $ — матрица весов второго слоя (размерность $ (h, k) $, где $ k $ — число выходных нейронов).  
- $ b_2 $ — вектор смещений второго слоя (размерность $ (1, k) $).  
- $ f $ — активационная функция (например, ReLU или сигмоида).  
- $ g $ — активация выходного слоя (например, Softmax для классификации).  

---

**1. Вычисление скрытого слоя**  
$$ H = f(X W_1 + b_1) $$
- $ X $ имеет размерность $ (d, n) $, $ W_1 $ — $ (d, h) $, $ b_1 $ — $ (1, h) $.  
- После матричного умножения $ X W_1 $ получаем размерность $ (d, h) $.  
- Добавляем $ b_1 $ и применяем функцию активации $ f $, получая матрицу активаций скрытого слоя $ H $ размерности $ (d, h) $.  

**2. Вычисление выходного слоя**  
$$ Y = g(H W_2 + b_2) $$
- $ H $ имеет размерность $ (d, h) $, $ W_2 $ — $ (h, k) $, $ b_2 $ — $ (1, k) $.  
- После умножения $ H W_2 $ получаем размерность $ (d, k) $.  
- Добавляем $ b_2 $ и применяем активацию $ g $, получая выходной вектор $ Y $ размерности $ (d, k) $.  

---

**Итоговая запись**  
$$ Y = g(f(X W_1 + b_1) W_2 + b_2) $$

---

### **Функция активации ReLU (Rectified Linear Unit)**  

**ReLU (Rectified Linear Unit)** — одна из самых популярных активационных функций в нейронных сетях. Она используется в скрытых слоях, чтобы добавлять нелинейность в модель.

---

**Формула ReLU**  
$$ f(x) = \max(0, x) $$
- Если $ x > 0 $, то $ f(x) = x $  
- Если $ x \leq 0 $, то $ f(x) = 0 $  

**ReLU обнуляет отрицательные значения и пропускает положительные без изменений.**

---

**ReLU — стандарт для глубоких нейросетей**.  


## Автокодировщик (autoencoder)

Автокодировщик (**Autoencoder, AE**) — это нейросеть, обучаемая на нормальных данных, чтобы восстанавливать их с минимальной ошибкой. Аномалии выявляются по значению ошибки восстановления (**reconstruction error**): если ошибка высокая, то входные данные, скорее всего, аномальные.

<!-- <img src="https://pub.mdpi-res.com/information/information-12-00238/article_deploy/html/images/information-12-00238-g001.png" alt="ae image" width="500"/> -->

<img src="https://upload.wikimedia.org/wikipedia/commons/3/37/Autoencoder_schema.png" alt="ae image" width="500"/>

---

### **1. Кодировщик (Encoder)**
Сжимает входные данные в скрытое представление (**latent space**) меньшей размерности.  

Пример:  
$$ h = f(Wx + b) $$
Где $ f $ — нелинейная активация (ReLU, LeakyReLU), \( W \) — веса сети.

### **2. Боттлнек (Latent Space)**
Самое узкое место в сети, где модель сохраняет только ключевые признаки данных.

### **3. Декодировщик (Decoder)**
Восстанавливает данные из латентного представления обратно в исходное пространство.

Пример:  
$$ \hat{x} = g(W'h + b') $$

### **4. Вычисление ошибки восстановления**
- **Mean Squared Error (MSE)**:  
  $$ \text{Loss} = \frac{1}{n} \sum (x - \hat{x})^2 $$
- Если ошибка выше порога, то это **аномалия**.

---

## **Сравнение с One-Class SVM**
| **Критерий**       | **Автокодировщик (AE)** | **One-Class SVM (OC-SVM)** |
|--------------------|------------------------|------------------------|
| **Тип модели** | Нейросеть с обучением | Метод SVM |
| **Пространство признаков** | Латентное представление | Гиперплоскость |
| **Адаптивность** | Гибкая, можно дообучать | Фиксированные границы |
| **Работа с нелинейностями** | Легко захватывает сложные структуры | Использует ядра, но ограничено |
| **Выявление аномалий** | По ошибке восстановления | По расстоянию от гиперплоскости |
| **Требования к данным** | Требуется большое количество нормальных данных | Работает даже с малым объемом |
| **Объясняемость** | Трудно интерпретировать | Легче понять границы класса |

---

- **AE лучше**, если аномалии имеют **сложные структуры** и нужно анализировать нелинейные зависимости.
- **OC-SVM лучше**, если данных **мало** и аномалии — это просто редкие отклонения.

Если данные промышленные и содержат **много параметров**, то **автокодировщик** обычно показывает **лучшие результаты**.


## Построение модели

Перед кодом разберем ключевые объекты PyTorch, которые используются в автокодировщике.

### `nn.Module`
Базовый класс для всех нейросетей в PyTorch.
Если мы наследуемся от `nn.Module`, то получаем:
- хранение параметров модели (`weights`, `bias`);
- режимы `model.train()` и `model.eval()`;
- удобный перенос на устройство (`model.to(device)`).

### `nn.Sequential`
Контейнер, который последовательно применяет слои друг за другом.
Удобен, когда архитектура линейная: `Layer1 -> Activation -> Layer2`.

### `nn.Linear`
Полносвязный слой: выполняет преобразование `y = xW^T + b`.
В автокодировщике один `Linear` сжимает признаки (encoder), второй восстанавливает их (decoder).

### `nn.ReLU`
Функция активации `max(0, x)`. Добавляет нелинейность, чтобы сеть могла моделировать более сложные зависимости.

### `forward`
Метод, который описывает прямой проход данных через модель.
Именно в `forward` мы задаем, как входной тензор проходит через encoder и decoder.


In [ ]:
n = data.shape[1] #

class Autoencoder(torch.nn.Module):
    def __init__(self, h):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n, h),
            nn.ReLU(),
            # nn.Linear(h, h),
        )
        self.decoder = nn.Sequential(
            nn.Linear(h, n),
            # nn.ReLU(),
            # nn.Linear(n, n),
        )

    def forward(self, sample):
        latent = self.encoder(sample)
        reconstructed = self.decoder(latent)
        return reconstructed

In [ ]:
model = Autoencoder(5)
model

In [ ]:
x = torch.tensor(X_train.iloc[10].values, dtype=torch.float32)[np.newaxis,:]
x_hat = model(x)

Входной вектор

In [ ]:
x

Выходной вектор

In [ ]:
x_hat

Посмотрим на веса нашей модели

In [ ]:
for name, param in model.encoder.state_dict().items():
    print(f"{name}: {param.shape}")
    print(param)

## Обучение сети

**Прямой проход (Forward Pass)**  
- Данные проходят через сеть, и каждый слой применяет свои веса и функции активации.  
- Получаем **выход модели** (предсказание).  

**Вычисление ошибки (Loss Calculation)**  
- Считаем **функцию потерь** (например, MSE, CrossEntropy).  
- Показывает, насколько сильно предсказание отличается от правильного ответа.

Среднеквадратичная ошибка (MSE) вычисляется как **среднее арифметическое квадратов разностей между предсказанными ($\hat{y}_i$) и реальными ($y_i$) значениями**:  

$$L = MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

Где:  
- $ N $ — количество примеров в выборке  
- $ y_i $ — истинное значение  
- $ \hat{y}_i $ — предсказанное значение модели  

---

**Обратное распространение ошибки (Backward Pass)**  
- Вычисляем **градиенты** ошибки по всем весам сети, начиная с выхода.  
- Используем **правило дифференцирования сложной функции** (chain rule), чтобы передавать градиенты назад по сети.  

**Обновление весов (Weight Update)**  
- Используем **оптимизатор** (например, `SGD`, `Adam`), чтобы обновить веса:  
  $$ w = w - \eta \cdot \frac{\partial L}{\partial w} $$
  где:  
  - $ w $ — веса сети  
  - $ \eta $ — скорость обучения (learning rate)  
  - $ \frac{\partial L}{\partial w} $ — градиент ошибки  


Заметим, что поскольку мы занимаемся реконструкцией, train/val у нас выступает как в роли входа для сети, так и в роли таргета

### Создаем `DataLoader` на обучающих и валидационных данных

`DataLoader` в PyTorch — это объект, который подает данные в модель по батчам (не весь датасет сразу).

Зачем он нужен:
- разбивает выборку на мини-батчи, что ускоряет и стабилизирует обучение;
- умеет перемешивать данные (`shuffle=True`) для train;
- упрощает цикл обучения: на каждой итерации получаем готовый `X_batch`.


Делим нашу обучающую выборку на обучающую и валидационную

In [ ]:
X_val = X_train.iloc[round(X_train.shape[0]*0.8):, :]
X_train = X_train.iloc[:round(X_train.shape[0]*0.8), :]

In [ ]:
BATCH_SIZE = 512 # Размер батча (количество примеров за раз)
# Разделение данных на батчи (mini-batches) при обучении нейросетей ускоряет и стабилизирует процесс градиентного спуска.

X_train_np = np.array(X_train.to_numpy(), dtype=np.float32, copy=True)
X_val_np = np.array(X_val.to_numpy(), dtype=np.float32, copy=True)

train_loader = torch.utils.data.DataLoader(
    torch.from_numpy(X_train_np), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    torch.from_numpy(X_val_np), batch_size=BATCH_SIZE, shuffle=False
)


### Обучение модели

In [ ]:
log = []

In [ ]:
HIDDEN_SIZE = 7
N_EPOCHS = 100
LEARNING_RATE = 0.001


model = Autoencoder(HIDDEN_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = F.mse_loss
train_losses = []
val_losses = []

for epoch in tqdm(range(N_EPOCHS)):
    model.train()
    train_losses_per_epoch = []
    for i, X_batch in enumerate(train_loader):
        X_batch = X_batch.to(torch.float32)
        optimizer.zero_grad()
        reconstructed = model(X_batch)
        loss = loss_fn(reconstructed, X_batch)
        loss.backward()
        optimizer.step()
        train_losses_per_epoch.append(loss.item())

    train_losses.append(np.mean(train_losses_per_epoch))

    model.eval()
    val_losses_per_epoch = []
    with torch.no_grad():
        for X_batch in val_loader:
            X_batch = X_batch.to(torch.float32)
            reconstructed = model(X_batch)
            loss = loss_fn(reconstructed, X_batch)
            val_losses_per_epoch.append(loss.item())

    val_losses.append(np.mean(val_losses_per_epoch))

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(np.arange(len(train_losses)), train_losses, label='Train')
plt.plot(np.arange(len(val_losses)), val_losses, label='Validation')

plt.xlabel('Эпоха')
plt.title('Среднеквадратичная ошибка (MSE loss)')
plt.legend()
plt.show()

In [ ]:
print('Loss в конце обучения')
print(f'На обучающей выборке: {train_losses[-1]:.5f}')
print(f'На валидационной выборке: {val_losses[-1]:.5f}')

## Оценка ошибки восстановления

In [ ]:
def get_reconstruction_error(df, model):
    x = torch.tensor(df.values, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        x_hat = model(x)
    err = F.mse_loss(x_hat, x, reduction='none').mean(axis=1)
    return err.detach().numpy()

val_reconstruction_err = get_reconstruction_error(X_val, model)
test_reconstruction_err = get_reconstruction_error(X_test, model)

data_test['reconstruction_err'] = test_reconstruction_err


In [ ]:
MAX_RECONSTRUCTION_ERR = np.inf

data_for_plot = data_test.copy()
data_for_plot.loc[data_for_plot['reconstruction_err'] > MAX_RECONSTRUCTION_ERR, ['reconstruction_err']] = MAX_RECONSTRUCTION_ERR

params_dropdown = widgets.Dropdown(
    options=data_test.columns,
    description='Параметр:',
    disabled=False,
    value=None
)

out = widgets.Output()
display(out)

with out:
    display(params_dropdown)

@out.capture()
def params_dropdown_eventhandler(change):

    clear_output()
    display(params_dropdown)
    # selected_param = selected_params_kks[list(selected_params_description).index(change.new)]


    fig, axes = plt.subplots(1, 1, figsize=(15,5))
    fig = plt.figure();



    plt.scatter(
        data_for_plot[change.new].index,
        data_for_plot[change.new].values,
        c = data_for_plot['reconstruction_err'],
        s=10, alpha=0.5);


    plt.title(f"{change.new} - {kks_to_description [change.new]}")
    plt.ylabel(change.new, fontsize=20);
    cbar = plt.colorbar()
    plt.set_cmap('viridis')
    cbar.ax.get_yaxis().labelpad = 20
    cbar.ax.set_ylabel('reconstruction_err', rotation=270, fontsize=20)
    display(fig)

params_dropdown.observe(params_dropdown_eventhandler, names='value')

### **ROC AUC — способность модели различать классы**
ROC AUC показывает **насколько хорошо модель различает нормальные и аномальные данные**, независимо от конкретного порога.  

**Как строится?**  
- **ROC-кривая** — график зависимости **True Positive Rate (TPR)** от **False Positive Rate (FPR)** при разных значениях порога.
  **TPR** показывает, какая доля положительных объектов была правильно классифицирована как положительные.
  **FPR** показывает, какая доля объектов отрицательного класса была ошибочно классифицирована как положительные.
- **AUC (Area Under Curve)** — площадь под этой кривой.

$$ TPR = \frac{TP}{TP + FN}, \quad FPR = \frac{FP}{FP + TN}$$

**Как интерпретировать?**  
✔ **AUC = 0.5** → модель не лучше случайного угадывания  
✔ **AUC → 1.0** → модель отлично различает аномалии и нормальные данные  
✔ **AUC < 0.5** → модель ошибочно классифицирует нормальные данные как аномалии  

**Важно:**  
- ROC AUC **не зависит от конкретного порога**, в отличие от F1-score.  
- Если классы сильно **дисбалансированы**, ROC AUC может быть **обманчиво высокой** (например, если аномалий мало, модель может просто предсказывать «норму» и получить высокий AUC).


In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(data_test['anomaly'], data_test['reconstruction_err'])
print(f"ROC AUC: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(data_test['anomaly'], data_test['reconstruction_err'])

plt.figure(figsize=(15, 5))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Линия случайного угадывания
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Anomaly Detection')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

### **PR-AUC — качество по редкому классу аномалий**
PR-кривая строится как зависимость **Precision** от **Recall** при разных порогах.

- `Precision = TP / (TP + FP)` — какая доля найденных аномалий действительно аномалии.
- `Recall = TP / (TP + FN)` — какую долю всех аномалий модель нашла.
- `PR-AUC` (или Average Precision) — площадь под PR-кривой. Чем выше, тем лучше.

Для задач поиска аномалий (редкий положительный класс) PR-AUC обычно информативнее, чем ROC-AUC.


In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

pr_auc = average_precision_score(data_test['anomaly'], data_test['reconstruction_err'])
precision, recall, pr_thresholds = precision_recall_curve(data_test['anomaly'], data_test['reconstruction_err'])

print(f'PR-AUC (Average Precision): {pr_auc:.4f}')

baseline = data_test['anomaly'].mean()
plt.figure(figsize=(15, 5))
plt.plot(recall, precision, color='darkorange', label=f'PR curve (AP = {pr_auc:.4f})')
plt.hlines(baseline, xmin=0, xmax=1, colors='gray', linestyles='--', label=f'Baseline = {baseline:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Anomaly Detection')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()


### Порог по валидации (без утечки из теста)

Порог выбираем **только по валидационной выборке** (без меток теста):
берем квантиль ошибки реконструкции и считаем, что значения выше порога — аномалии.


In [ ]:
VAL_ANOMALY_RATE = 0.2  # ожидаемая доля аномалий во валидации
optimal_threshold = np.quantile(val_reconstruction_err, 1 - VAL_ANOMALY_RATE)

print(f'Квантиль порога: {1 - VAL_ANOMALY_RATE:.2%}')
print(f'Порог по валидации: {optimal_threshold:.6f}')


### Accuracy, Precision, Recall, F1

In [ ]:
data_test['anomaly_ae'] = (data_test['reconstruction_err'] > optimal_threshold).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = data_test['anomaly'].values
y_pred = data_test['anomaly_ae'].values

# roc_auc и pr_auc уже рассчитаны в ячейках выше
model_metrics = {
    'roc_auc': float(roc_auc),
    'pr_auc': float(pr_auc),
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'precision': float(precision_score(y_true, y_pred, average='binary', zero_division=0)),
    'recall': float(recall_score(y_true, y_pred, average='binary', zero_division=0)),
    'f1': float(f1_score(y_true, y_pred, average='binary', zero_division=0)),
}

print(f"accuracy - {model_metrics['accuracy']*100:0.2f}%")
print(f"precision - {model_metrics['precision']*100:0.2f}%")
print(f"recall - {model_metrics['recall']*100:0.2f}%")
print(f"F1 - {model_metrics['f1']*100:0.2f}%")
print(f"ROC AUC - {model_metrics['roc_auc']:.4f}")
print(f"PR AUC - {model_metrics['pr_auc']:.4f}")


In [ ]:
log_row = {
    'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'config': {
        'hidden_size': HIDDEN_SIZE,
        'n_epochs': N_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'val_anomaly_rate': VAL_ANOMALY_RATE,
        'threshold': round(float(optimal_threshold), 3),
    },
    'metrics': {
        'roc_auc': round(model_metrics['roc_auc'], 4),
        'pr_auc': round(model_metrics['pr_auc'], 4),
        'accuracy_pct': round(model_metrics['accuracy'] * 100, 2),
        'precision_pct': round(model_metrics['precision'] * 100, 2),
        'recall_pct': round(model_metrics['recall'] * 100, 2),
        'f1_pct': round(model_metrics['f1'] * 100, 2),
        'train_loss_last': round(float(train_losses[-1]), 3),
        'val_loss_last': round(float(val_losses[-1]), 3),
        'reconstruction_err_test_mean': round(float(np.mean(data_test['reconstruction_err'])), 3),
        'reconstruction_err_test_std': round(float(np.std(data_test['reconstruction_err'])), 3),
    }
}

log.append(log_row)
print('Запись в лог добавлена')
log_row


In [ ]:
log

## Как повысить точность модели

Практические шаги для улучшения качества детекции аномалий:
- сделать архитектуру автоэнкодера чуть глубже и подобрать `HIDDEN_SIZE`;
- настроить гиперпараметры обучения: `learning_rate`, `batch_size`, `N_EPOCHS`, добавить early stopping;
- сравнить MinMaxScaler и StandardScaler и оставить вариант с лучшей валидационной метрикой;
- отслеживать PR-AUC и F1 вместе с ROC-AUC, так как при дисбалансе классов они информативнее.
